# 🏥 Predicción de Readmisión Hospitalaria en Pacientes Diabéticos
## ACIF104 — Aprendizaje Automático | UNAB 2026

---

**Objetivo:** Este cuaderno implementa el pipeline completo del proyecto:
1. Carga y preprocesamiento del dataset
2. Análisis exploratorio de datos (EDA)
3. Entrenamiento de modelos de ML clásico (LR, RF, SVM)
4. Entrenamiento de red neuronal MLP con PyTorch
5. Explicabilidad con Tree SHAP
6. Sistema de predicción interactivo

**Dataset:** `hospital_readmissions.csv` (25.000 registros, 16 predictores + 1 variable objetivo)

---
⚠️ **Instrucción inicial:** Sube el archivo `hospital_readmissions.csv` usando la celda de carga que aparece a continuación.

## ⚙️ PASO 1 — Instalación de dependencias
Ejecuta esta celda primero. Puede tardar 1–2 minutos.

In [ ]:
# Instalación de librerías necesarias (solo la primera vez)
import sys

print('Instalando dependencias...')
!pip install -q shap imbalanced-learn torch torchvision --upgrade
print('✓ Dependencias instaladas correctamente.')

: 

## 📁 PASO 2 — Carga del dataset

**En Google Colab:** Ejecuta la celda siguiente. Aparecerá un botón "Elegir archivos". Selecciona `hospital_readmissions.csv`.

**En JupyterLab local:** Sube el archivo a la misma carpeta que este cuaderno y ajusta la variable `CSV_PATH`.

In [ ]:
import os

# ── Detectar entorno: Google Colab o JupyterLab local ────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('📂 Selecciona el archivo hospital_readmissions.csv:')
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
    print(f'✓ Archivo cargado: {CSV_PATH}')
else:
    # JupyterLab local: ajusta la ruta si es necesario
    CSV_PATH = 'hospital_readmissions.csv'
    if not os.path.exists(CSV_PATH):
        print(f'✗ Archivo no encontrado: {CSV_PATH}')
        print('  Coloca hospital_readmissions.csv en la misma carpeta que este cuaderno.')
    else:
        print(f'✓ Archivo encontrado: {CSV_PATH}')

## 📦 PASO 3 — Importaciones y configuración global

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, average_precision_score, precision_recall_curve
)
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import shap

# Semillas para reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paleta de colores del proyecto
C = {
    'p': '#1F3864', 's': '#2E75B6', 'a': '#5BA3D9',
    'l': '#A8D1F0', 'r': '#C00000', 'g': '#595959'
}
sns.set_theme(style='whitegrid')

# Nombres legibles de las variables
FEATURES = [
    'time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications',
    'n_outpatient', 'n_inpatient', 'n_emergency', 'age_enc',
    'glucose_test_enc', 'A1Ctest_enc', 'change_enc', 'diabetes_med_enc',
    'medical_specialty_enc', 'diag_1_enc', 'diag_2_enc', 'diag_3_enc'
]
FEAT_LABELS = [
    'Días en hospital', 'Lab. proc.', 'Procedimientos', 'Medicamentos',
    'Visitas ambulat.', 'Ingresos previos', 'Urgencias', 'Edad',
    'Test glucosa', 'Test A1C', 'Cambio medicam.', 'Med. diabetes',
    'Especialidad', 'Diagnóstico 1', 'Diagnóstico 2', 'Diagnóstico 3'
]

print('✓ Librerías importadas correctamente.')
print(f'  PyTorch: {torch.__version__}')
print(f'  SHAP:    {shap.__version__}')
print(f'  GPU disponible: {torch.cuda.is_available()}')

## 🔬 PASO 4 — Carga y preprocesamiento de datos

In [ ]:
# ── Carga ─────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
display(df.head(3))

# ── Calidad de datos ──────────────────────────────────────────
print('\n=== Valores nulos por variable ===')
print(df.isnull().sum())

print('\n=== Distribución de la variable objetivo ===')
vc = df['readmitted'].value_counts()
for k, v in vc.items():
    print(f'  {k}: {v:,} ({v/len(df)*100:.1f} %)')

# ── Codificación de variables categóricas ─────────────────────
le = LabelEncoder()
CAT_COLS = ['age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3',
            'glucose_test', 'A1Ctest', 'change', 'diabetes_med']
for col in CAT_COLS:
    df[col + '_enc'] = le.fit_transform(df[col])

# Variable objetivo binaria
df['target'] = (df['readmitted'] == 'yes').astype(int)

# ── Análisis de valores atípicos (IQR) ───────────────────────
NUM_COLS = ['time_in_hospital', 'n_lab_procedures', 'n_procedures',
            'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency']
print('\n=== Valores atípicos detectados (método IQR) ===')
for col in NUM_COLS:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col:<22}: {n_out:4d} ({n_out/len(df)*100:.2f} %)')

print('\n✓ Preprocesamiento completado.')

## 📊 PASO 5 — Análisis Exploratorio de Datos (EDA)

In [ ]:
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Gráfico 1: Distribución de la variable objetivo ───────────
ax1 = fig.add_subplot(gs[0, 0])
vals = df['readmitted'].value_counts().reindex(['no', 'yes'])
bars = ax1.bar(vals.index, vals.values, color=[C['p'], C['s']],
               edgecolor='white', linewidth=1.5, width=0.5)
for bar, v in zip(bars, vals.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
             f'{v:,}\n({v/len(df)*100:.1f} %)',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color=C['p'])
ax1.set_title('Distribución de la variable objetivo', fontsize=12,
               fontweight='bold', color=C['p'])
ax1.set_xlabel('Readmitido'); ax1.set_ylabel('Frecuencia')
ax1.set_ylim(0, 16000)

# ── Gráfico 2: n_inpatient por clase (predictor clave) ────────
ax2 = fig.add_subplot(gs[0, 1])
data_n = df[df['readmitted']=='no']['n_inpatient'].values
data_y = df[df['readmitted']=='yes']['n_inpatient'].values
bp = ax2.boxplot([data_n, data_y], tick_labels=['No readmitido', 'Readmitido'],
                  patch_artist=True,
                  medianprops=dict(color='white', linewidth=2),
                  flierprops=dict(marker='o', markerfacecolor=C['r'],
                                  markersize=2, alpha=0.3))
bp['boxes'][0].set_facecolor(C['p'])
bp['boxes'][1].set_facecolor(C['s'])
ax2.set_title('Ingresos previos (n_inpatient) — Predictor clave',
               fontsize=12, fontweight='bold', color=C['p'])
ax2.set_ylabel('Hospitalizaciones previas')

# ── Gráfico 3: Distribución de edad por clase ─────────────────
ax3 = fig.add_subplot(gs[1, 0])
age_order = ['[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
ct = pd.crosstab(df['age'], df['readmitted']).reindex(age_order)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct[['no', 'yes']].plot(kind='bar', ax=ax3, color=[C['p'], C['s']],
                            edgecolor='white', width=0.65)
ax3.set_title('Tasa de readmisión por grupo etario',
               fontsize=12, fontweight='bold', color=C['p'])
ax3.set_xlabel('Grupo de edad'); ax3.set_ylabel('Porcentaje (%)')
ax3.tick_params(axis='x', rotation=30)
ax3.legend(['No readmitido', 'Readmitido'], fontsize=9)

# ── Gráfico 4: Estadística descriptiva de variables numéricas ─
ax4 = fig.add_subplot(gs[1, 1])
means_n = df[df['readmitted']=='no'][NUM_COLS].mean()
means_y = df[df['readmitted']=='yes'][NUM_COLS].mean()
x = np.arange(len(NUM_COLS))
w = 0.38
ax4.bar(x - w/2, means_n.values, w, label='No readmitido', color=C['p'], alpha=0.85)
ax4.bar(x + w/2, means_y.values, w, label='Readmitido',    color=C['s'], alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(['Días', 'Lab', 'Proc', 'Medic', 'Ambul', 'Ingr', 'Urg'],
                     fontsize=9)
ax4.set_title('Media de variables clínicas por clase',
               fontsize=12, fontweight='bold', color=C['p'])
ax4.set_ylabel('Media'); ax4.legend(fontsize=9)

# ── Gráfico 5: Mapa de calor de correlaciones ─────────────────
ax5 = fig.add_subplot(gs[2, :])
num_enc = NUM_COLS + ['age_enc', 'glucose_test_enc', 'A1Ctest_enc',
                       'change_enc', 'diabetes_med_enc', 'target']
corr = df[num_enc].corr()
labels_short = ['Días','Lab','Proc','Medic','Ambul','Ingr','Urg',
                 'Edad','Glucosa','A1C','Cambio','Med.diab','Readmitido']
sns.heatmap(corr, ax=ax5, cmap=sns.diverging_palette(220, 10, as_cmap=True),
            center=0, annot=True, fmt='.2f', linewidths=0.4,
            annot_kws={'size': 7.5},
            xticklabels=labels_short, yticklabels=labels_short,
            mask=np.triu(np.ones_like(corr, dtype=bool)))
ax5.set_title('Mapa de calor de correlaciones entre variables',
               fontsize=12, fontweight='bold', color=C['p'])

plt.suptitle('Análisis Exploratorio de Datos — Hospital Readmissions',
             fontsize=14, fontweight='bold', color=C['p'], y=1.01)
plt.savefig('eda_completo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ EDA completado. Figura guardada como eda_completo.png')